In [8]:
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
# Cell 1: Create the Full Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Reload the raw columns
df = pd.read_csv('../data/raw/StudentsPerformance.csv', sep=',') # Or appropriate separator
X = df.drop('math score', axis=1) # Replace with your target name
y = df['math score']

# Define column types
num_cols = X.select_dtypes(exclude=['object']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Create a full preprocessing and modeling pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
        ])),
    ('regressor', GradientBoostingRegressor(learning_rate=0.1, max_depth=3, n_estimators=100))
])

# Fit on full dataset (for final production model)
pipeline.fit(X, y)
print("Pipeline trained successfully!")

Pipeline trained successfully!


In [10]:
# Cell 2: Log to Model Registry
import mlflow

mlflow.set_tracking_uri("sqlite:///../mlflow1.db")
mlflow.set_experiment("Student_Performance_Prediction")

with mlflow.start_run(run_name="Production_Pipeline_Run"):
    # Log the entire pipeline
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        registered_model_name="StudentMathModel"
    )
    print("✅ Model registered as 'StudentMathModel' in MLflow Registry!")

2026/04/15 11:51:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/15 11:51:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'StudentMathModel'.
Created version '1' of model 'StudentMathModel'.


✅ Model registered as 'StudentMathModel' in MLflow Registry!


In [11]:
import os
import joblib

# 1. Define local path
os.makedirs('../models', exist_ok=True)
local_model_path = '../models/student_math_model.pkl'

# 2. Save locally using joblib (Standard professional way)
joblib.dump(pipeline, local_model_path)
print(f"✅ Model saved locally at: {local_model_path}")

✅ Model saved locally at: ../models/student_math_model.pkl
